## Main

In [1]:
#| default_exp mass.massmain_module

In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
#| export

from anthropmass.mass.com_calculation_module import *
from anthropmass.mass.com_calculation_module import *
from anthropmass.mass.volume_calculation_module import *
from anthropmass.mass.plot_body_module import *
from anthropmass.mass.inertial_calc_module import *
from anthropmass.mass.massclauser_module import *
from anthropmass.mass.measurements_heights_module import get_measurements, get_heights
import pandas as pd



## Main
This function gathers variables and persents the in print lines. Prints, estimated heights and total masses, segment masses for both methods, positions of COM and joint centers, and moments of inertia. It also calls on the function to plot the 2D figure and plots the figure using the same inputs as for the parameters mentioned.

In [ ]:
#| export
def main(Ansur, inputweight, inputheight, gender):
    

    #Gathers librarys from other function
    heights = get_heights(Ansur, inputheight).iloc[0]
    pointsJC, com = get_joint_and_com_points(Ansur, inputheight, gender)
    weights = clauser(Ansur.iloc[0], inputweight, inputheight)
    weightsZat = zatsiorsky(inputweight, inputheight)
    inertia = calculate_inertia(Ansur, inputweight, inputheight)

    






    print("..............................................................................................................")

    #Prints information about height and total estimated weight
    print("RESULTS:")
    print("Estimated Height:", heights["TotalH"], "m")
    print("Actual Height:", inputheight/1000, "m")
    print("Total Estimated Weight (kg) using Clauser regression model:",weights['mTOT'],"kg" )
    print("Total Estimated Weight using Zatriosky regression model:",weightsZat['mTOTzat'],"kg" )




    print("..............................................................................................................")




    #Prints the COM positions and aranges in dataframe
    segments = [
        'Head', 'Upper Trunk', 'Middle Trunk', 'Lower Trunk', 'Upper Arm',
        'Lower Arm', 'Hand', 'Thigh', 'Shank', 'Foot'
    ]

    #Please note: COM positions are stored like 2D coordinates (y, z). Since all of them (exept foot) are placed on (x = 0) 
    #Ex: com("U(Upper)T(Trunk)M(Mass)C(Center)")[0] gives y coordinate for upper trunk
    positions_df = pd.DataFrame(index=['X', 'Y', 'Z'], columns=segments)

    positions_df.loc[:, 'Head'] = [0, round(com['HeMC'][0], 3), round(com['HeMC'][1], 3)]
    positions_df.loc[:, 'Upper Trunk'] = [0, round(com['UTMC'][0], 3), round(com['UTMC'][1], 3)]
    positions_df.loc[:, 'Middle Trunk'] = [0, round(com['MTMC'][0], 3), round(com['MTMC'][1], 3)]
    positions_df.loc[:, 'Lower Trunk'] = [0, round(com['LTMC'][0], 3), round(com['LTMC'][1], 3)]
    positions_df.loc[:, 'Upper Arm'] = [0, round(com['RUAMC'][0], 3), round(com['RUAMC'][1], 3)]
    positions_df.loc[:, 'Lower Arm'] = [0, round(com['RLAMC'][0], 3), round(com['RLAMC'][1], 3)]
    positions_df.loc[:, 'Hand'] = [0, round(com['RHMC'][0], 3), round(com['RHMC'][1], 3)]
    positions_df.loc[:, 'Thigh'] = [0, round(com['RTHMC'][0], 3), round(com['RTHMC'][1], 3)]
    positions_df.loc[:, 'Shank'] = [0, round(com['RSHMC'][0], 3), round(com['RSHMC'][1], 3)]
    positions_df.loc[:, 'Foot'] = [round(com['RFMCx'], 3), round(com['RFMC'][0], 3), round(com['RFMC'][1], 3)]

    print("\nCenter of mass positions:\n")
    print(positions_df.to_string())


    print("..............................................................................................................")


    #Prints the joint center positions and aranges in dataframe
    #Please note: joint positions are stored like 2D coordinates (y, z). Since all of them are placed on (x = 0) 

    joints = ["Shoulder", "Elbow", "Wrist", "Hip", "Knee", "Ankle"]

    #Create the DataFrame
    jc_positions_df = pd.DataFrame(index=['X', 'Y', 'Z'], columns=joints)

    #Fill the table with absolute Y values 
    for joint in joints:
        coords = pointsJC[joint]
        y_depth = round(abs(coords[0]), 3)   # take absolute value of Y (distance from centerline)
        z_height = round(coords[1], 3)
        jc_positions_df.loc[:, joint] = [0, y_depth, z_height]

    # Display the table
    print("\nJoint center positions:\n")
    print(jc_positions_df.to_string())


    print("..............................................................................................................")







    #Print masses from clausers estimation of mass
    #This only useses one segment for trunk
    #Segment masses correspond to 2D and 3D visualisation
    mass_segments = [
        "Head", "Trunk Total", "Upper Trunk", "Middle Trunk", "Lower Trunk",
        "Upper Arm", "Forearm", "Hand", "Thigh", "Shank", "Foot"
    ]

    # Create mass comparison table
    mass_df = pd.DataFrame(index=["Clauser", "Zatsiorsky"], columns=mass_segments)

    # --- Clauser Values ---
    mass_df.loc["Clauser", "Head"] = round(weights["mHe"], 3)
    mass_df.loc["Clauser", "Trunk Total"] = round(weights["mTr"], 3)
    mass_df.loc["Clauser", "Upper Trunk"] = "---"
    mass_df.loc["Clauser", "Middle Trunk"] = "---"
    mass_df.loc["Clauser", "Lower Trunk"] = "---"
    mass_df.loc["Clauser", "Upper Arm"] = round(weights["mUA"], 3)
    mass_df.loc["Clauser", "Forearm"] = round(weights["mLA"], 3)
    mass_df.loc["Clauser", "Hand"] = round(weights["mH"], 3)
    mass_df.loc["Clauser", "Thigh"] = round(weights["mTh"], 3)
    mass_df.loc["Clauser", "Shank"] = round(weights["mS"], 3)
    mass_df.loc["Clauser", "Foot"] = round(weights["mF"], 3)








    #Print masses from zatsiorsky estimation of mass
    #This useses 3 segments for trunk
    #Segment masses DOES NOT correspond to 2D and 3D visualisation
    # --- Zatsiorsky Values ---
    trunk_total_zat = (
        weightsZat["mUppTrzat"] +
        weightsZat["mMidTrzat"] +
        weightsZat["mLowTrzat"]
    )

    mass_df.loc["Zatsiorsky", "Head"] = round(weightsZat["mHezat"], 3)
    mass_df.loc["Zatsiorsky", "Trunk Total"] = round(trunk_total_zat, 3)
    mass_df.loc["Zatsiorsky", "Upper Trunk"] = round(weightsZat["mUppTrzat"], 3)
    mass_df.loc["Zatsiorsky", "Middle Trunk"] = round(weightsZat["mMidTrzat"], 3)
    mass_df.loc["Zatsiorsky", "Lower Trunk"] = round(weightsZat["mLowTrzat"], 3)
    mass_df.loc["Zatsiorsky", "Upper Arm"] = round(weightsZat["mUAzat"], 3)
    mass_df.loc["Zatsiorsky", "Forearm"] = round(weightsZat["mLAzat"], 3)
    mass_df.loc["Zatsiorsky", "Hand"] = round(weightsZat["mHzat"], 3)
    mass_df.loc["Zatsiorsky", "Thigh"] = round(weightsZat["mThzat"], 3)
    mass_df.loc["Zatsiorsky", "Shank"] = round(weightsZat["mSzat"], 3)
    mass_df.loc["Zatsiorsky", "Foot"] = round(weightsZat["mFzat"], 3)

    # --- Display the table ---
    print("\nSegment Mass Comparison (Kg):\n")
    print(mass_df.to_string())


    print("..............................................................................................................")


    # Build a DataFrame for inertia that mirrors the style of your COM and mass tables
    inertia_segments = [
        "Upper Trunk", "Middle Trunk", "Lower Trunk",
        "Thigh", "Shank", "Foot",
        "Upper Arm", "Lower Arm", "Hand", "Head"
    ]

    inertia_df = pd.DataFrame(index=inertia_segments, columns=["Ixx", "Iyy", "Izz"])

    # Fill in each segment explicitly
    inertia_df.loc["Upper Trunk"]  = [inertia["Ixx_UT"], inertia["Iyy_UT"], inertia["Izz_UT"]]
    inertia_df.loc["Middle Trunk"] = [inertia["Ixx_MT"], inertia["Iyy_MT"], inertia["Izz_MT"]]
    inertia_df.loc["Lower Trunk"]  = [inertia["Ixx_LT"], inertia["Iyy_LT"], inertia["Izz_LT"]]

    inertia_df.loc["Thigh"]        = [inertia["Ixx_T"],  inertia["Iyy_T"],  inertia["Izz_T"]]
    inertia_df.loc["Shank"]        = [inertia["Ixx_S"],  inertia["Iyy_S"],  inertia["Izz_S"]]
    inertia_df.loc["Foot"]         = [inertia["Ixx_F"],  inertia["Iyy_F"],  inertia["Izz_F"]]

    inertia_df.loc["Upper Arm"]    = [inertia["Ixx_UA"], inertia["Iyy_UA"], inertia["Izz_UA"]]
    inertia_df.loc["Lower Arm"]    = [inertia["Ixx_LA"], inertia["Iyy_LA"], inertia["Izz_LA"]]
    inertia_df.loc["Hand"]         = [inertia["Ixx_Ha"], inertia["Iyy_Ha"], inertia["Izz_Ha"]]
    inertia_df.loc["Head"]         = [inertia["Ixx_H"],  inertia["Iyy_H"],  inertia["Izz_H"]]

    # Round to four decimals for readability
    inertia_df = inertia_df.astype(float).round(4)

    # Display the table
    print("\nSegment Moments of Inertia (kg·m²)\n")
    print(inertia_df.to_string())


    print("..............................................................................................................")



    # Draw the plot
    plot_body(Ansur, inputheight, gender)

    

    
    


In [12]:

import nbdev; nbdev.nbdev_export()
